# OHAYO - QLoRA Training
Code cell 1-3: chuan bi/kiem moi truong. Code cell 4-5: GPU smoke 2 optimizer steps va xuat minh chung. Run All luc nay SE chay smoke that, KHONG chay pilot 50.
Chon GPU T4, bat Internet, giu Dataset Private qlora-environment-input. Khong can Secret, ngrok, BGE hay checkpoint inference. Khong reset phien sau khi tai base neu con can chay tiep.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib, io, os, stat, sys
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/qlora-preparation')
EXPECTED = {'tools.bundle': '7fd1a989602f72951e3e7626f605d6509234183e04ef165f21e8eca35fc15b55', 'pilot.bundle': '2b44e35f7386b506955e9fad29541490c052129a8af6ae0eef7e4f0b7f7c0707'}
def obtain(name):
    candidates = []
    for path in INPUT.rglob(name):
        if path.is_file() and path.stat().st_size <= 64 * 1024**2:
            raw = path.read_bytes()
            if hashlib.sha256(raw).hexdigest() == EXPECTED[name]:
                candidates.append(raw)
    if not candidates:
        for path in INPUT.rglob('qlora-environment-input.zip'):
            with ZipFile(path) as archive:
                info = archive.getinfo(name)
                assert info.file_size <= 64 * 1024**2, 'INPUT_TOO_LARGE'
                raw = archive.read(info)
                if hashlib.sha256(raw).hexdigest() == EXPECTED[name]:
                    candidates.append(raw)
    assert len(candidates) == 1, f'{name}: can dung 1 Input khop checksum, tim thay {len(candidates)}'
    return candidates[0]
def unpack(raw, destination):
    destination = destination.resolve()
    with ZipFile(io.BytesIO(raw)) as archive:
        infos = archive.infolist()
        names = [entry.filename for entry in infos]
        assert len(infos) <= 200 and len(names) == len(set(n.casefold() for n in names)), 'ZIP_ENTRY_SET'
        assert sum(entry.file_size for entry in infos) <= 64 * 1024**2, 'ZIP_TOO_LARGE'
        assert archive.testzip() is None, 'ZIP_CRC'
        for entry in infos:
            name = entry.filename
            assert not entry.is_dir() and not stat.S_ISLNK(entry.external_attr >> 16), 'ZIP_MEMBER_TYPE'
            assert '\\' not in name and ':' not in name and not name.startswith('/'), 'ZIP_PATH'
            path = (destination / name).resolve()
            assert path.is_relative_to(destination) and path != destination, 'ZIP_TRAVERSAL'
            content = archive.read(entry)
            if path.exists():
                assert path.is_file() and path.read_bytes() == content, 'EXISTING_INPUT_DIFFERS'
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with path.open('xb') as stream:
                    stream.write(content)
        assert {p.relative_to(destination).as_posix() for p in destination.rglob('*') if p.is_file()} == set(names), 'UNEXPECTED_EXTRACTED_FILES'
TOOLS = WORK / 'tools'
HANDOFF = WORK / 'handoff'
unpack(obtain('tools.bundle'), TOOLS)
unpack(obtain('pilot.bundle'), HANDOFF)
print('INPUTS_VERIFIED - code va pilot da duoc kiem checksum va giai nen.')


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib, json, os, shutil, subprocess
SINGLE_GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
VENV = Path('/kaggle/working/qlora-env')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(VENV), 'install', '--disable-pip-version-check', 'pip==' + version('pip')], check=True)

# Corrected environment checker, copied into a new runtime tree, not locked Input.
SETUP_SOURCE = '"""GPU environment checks only: no Qwen weights, LoRA training or optimizer steps."""\nimport argparse\nimport gc\nfrom importlib.metadata import version\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom contextlib import redirect_stderr, redirect_stdout\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import environment, finish_manifest, output_guard, write_json\n\n\ndef check_hardware(torch, working):\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(torch.version.cuda is not None, \'CUDA_TORCH_REQUIRED_NOT_CPU_WHEEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    disk = shutil.disk_usage(working)\n    # This phase writes small reports/tensors only; base-cache budgeting belongs\n    # to the weight-loading phase and must not be reported as verified here.\n    require(disk.free >= 1024 ** 3, \'NEED_1_GIB_FREE_FOR_ENVIRONMENT_CHECK\')\n    return dict(device=torch.cuda.get_device_name(0), free_vram_bytes=free,\n                total_vram_bytes=total, free_working_disk_bytes=disk.free,\n                free_working_disk_gib=round(disk.free / 1024 ** 3, 3),\n                training_storage_check=\'PENDING_BEFORE_BASE_WEIGHT_LOADING\')\n\n\ndef probe_nf4(torch):\n    import bitsandbytes as bnb\n    # A random frozen layer tests CUDA kernels, not Qwen or an optimizer update.\n    torch.manual_seed(42)\n    with torch.cuda.device(0):\n        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=torch.float16,\n                                  compress_statistics=True, quant_type=\'nf4\')\n        layer.requires_grad_(False)\n        layer = layer.to(\'cuda:0\')\n        sample = torch.randn(2, 64, device=\'cuda:0\', dtype=torch.float16, requires_grad=True)\n        output = layer(sample)\n        require(output.dtype == torch.float16 and torch.isfinite(output).all().item(), \'NF4_FP16_FORWARD\')\n        output.float().square().mean().backward()\n        require(sample.grad is not None and torch.isfinite(sample.grad).all().item()\n                and sample.grad.abs().sum().item() > 0, \'NF4_INPUT_GRADIENT\')\n        state = layer.weight.quant_state\n        require(layer.weight.device.type == \'cuda\' and state.quant_type == \'nf4\'\n                and state.nested, \'NF4_DOUBLE_QUANTIZATION\')\n        torch.cuda.synchronize(0)\n        del sample, output, layer\n    gc.collect()\n    torch.cuda.empty_cache()\n    return dict(nf4_cuda_forward=\'PASSED\', fp16_compute=\'PASSED\', input_backward=\'PASSED\',\n                double_quantization=True, optimizer_steps=0, qwen_weights_loaded=False)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\',\n                      PYTHONDONTWRITEBYTECODE=\'1\', WANDB_DISABLED=\'true\')\n    config = read_json(args.config)\n    output = output_guard(args.output, [args.bundle, args.config])\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    try:\n        with (output / \'setup-output.txt\').open(\'w\', encoding=\'utf-8\') as stream, \\\n             redirect_stdout(stream), redirect_stderr(stream):\n            import torch\n            from packaging.version import Version\n            write_json(output / \'environment.json\', environment())\n            require(Version(torch.__version__.split(\'+\')[0]) >= Version(\'2.6\'), \'TORCH_2_6_OR_NEWER_REQUIRED\')\n            for name, expected in config[\'dependencies\'].items():\n                require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n            hardware = check_hardware(torch, output)\n            from transformers import Trainer, TrainingArguments\n            from peft import LoraConfig, prepare_model_for_kbit_training\n            from accelerate import Accelerator\n            probe = probe_nf4(torch)\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            for rows in data.values():\n                for offset in range(0, len(rows), 4):\n                    collator(rows[offset:offset + 4])\n            write_json(output / \'environment-check.json\', dict(\n                status=\'ENVIRONMENT_READY_FOR_SMOKE\', hardware=hardware, kernel_probe=probe,\n                records_checked=sum(map(len, data.values())), train=len(data[\'train\']),\n                validation=len(data[\'validation\']), qwen_weights_loaded=False,\n                qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n            status = \'ENVIRONMENT_READY_FOR_SMOKE\'\n    except BaseException as exc:\n        error = type(exc).__name__\n        write_json(output / \'environment-check.json\', dict(status=\'FAILED\', error_type=error,\n                   error=str(exc), qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=\'environment-only\', config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'
SETUP_SHA256 = 'cc9810b4414fddabf62757891e59f1daf22c57730bee1dd299d0da68c39f0d8a'
assert hashlib.sha256(SETUP_SOURCE.encode('utf-8')).hexdigest() == SETUP_SHA256
ORIGINAL_TOOLS = WORK / 'tools'
RUN_TOOLS = WORK / ('tools-environment-' + SETUP_SHA256[:12])
if not RUN_TOOLS.exists():
    shutil.copytree(ORIGINAL_TOOLS, RUN_TOOLS)
    (RUN_TOOLS / 'scripts/qlora/setup.py').write_bytes(SETUP_SOURCE.encode('utf-8'))
for original in ORIGINAL_TOOLS.rglob('*'):
    if original.is_file():
        relative = original.relative_to(ORIGINAL_TOOLS)
        actual = RUN_TOOLS / relative
        if relative.as_posix() == 'scripts/qlora/setup.py':
            assert hashlib.sha256(actual.read_bytes()).hexdigest() == SETUP_SHA256
        else:
            assert actual.read_bytes() == original.read_bytes(), 'RUNTIME_COPY_DIFFERS'
assert {p.relative_to(RUN_TOOLS).as_posix() for p in RUN_TOOLS.rglob('*') if p.is_file()} == {p.relative_to(ORIGINAL_TOOLS).as_posix() for p in ORIGINAL_TOOLS.rglob('*') if p.is_file()}
TOOLS = RUN_TOOLS
REPORT = WORK / 'reports' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
command = [sys.executable, '-B', '-m', 'scripts.qlora.kaggle', '--bundle', str(HANDOFF), '--profile', 'smoke', '--environment-only', '--output', str(REPORT)]
subprocess.run(command, cwd=TOOLS, env=SINGLE_GPU_ENV, check=True)


In [ ]:
import json
result = json.loads((REPORT / 'environment-check.json').read_text())
print(json.dumps(result, ensure_ascii=False, indent=2))
print('Bao cao:', REPORT)
assert result['status'] == 'ENVIRONMENT_READY_FOR_SMOKE', 'Moi truong chua dat; gui loi de kiem tra.'
print('Moi truong da dat. Chua chay smoke/pilot Qwen. Gui output cell nay de xac nhan buoc tiep theo.')


## Buoc 4 - GPU smoke
Hai cell tiep theo tai base dung revision (neu chua cache), chay 2 optimizer steps, kiem va dong goi minh chung. Khong chay pilot 50. Chi chay sau khi moi truong dat.


In [ ]:
# Step 4: actual Qwen 7B GPU smoke, exactly 2 optimizer steps.
import os, time
assert json.loads((REPORT / 'environment-check.json').read_text())['status'] == 'ENVIRONMENT_READY_FOR_SMOKE'
if 'SMOKE_PROCESS' in globals() and SMOKE_PROCESS.poll() is None:
    raise RuntimeError('A smoke process is already running; do not launch a second one.')
SMOKE_SOURCES = {'data.py': '"""Authenticate the locked pilot and recompute its tokenizer boundaries on CPU."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport unicodedata\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef read_json(path):\n    return loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef loads(text):\n    def pairs(items):\n        result = {}\n        for key, value in items:\n            require(key not in result, \'DUPLICATE_JSON_KEY\')\n            result[key] = value\n        return result\n    def invalid(value):\n        raise ValueError(\'NONFINITE_JSON\')\n    return json.loads(text, object_pairs_hook=pairs, parse_constant=invalid)\n\n\ndef canonical(value):\n    def normalize(item):\n        if isinstance(item, str):\n            return unicodedata.normalize(\'NFC\', item)\n        if isinstance(item, list):\n            return [normalize(v) for v in item]\n        if isinstance(item, dict):\n            return {k: normalize(v) for k, v in item.items()}\n        return item\n    return json.dumps(normalize(value), ensure_ascii=False, sort_keys=True,\n                      separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef sha_file(path):\n    sha = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            sha.update(block)\n    return sha.hexdigest()\n\n\ndef within(root, name):\n    root = Path(root).resolve()\n    require(isinstance(name, str) and \'\\\\\' not in name and \':\' not in name,\n            \'UNSAFE_MEMBER_PATH\')\n    path = (root / name).resolve()\n    require(path.is_relative_to(root) and path != root, \'MEMBER_OUTSIDE_ROOT\')\n    return path\n\n\ndef read_rows(path):\n    rows = []\n    for line in Path(path).read_text(encoding=\'utf-8\').splitlines():\n        require(bool(line.strip()), \'EMPTY_JSONL_ROW\')\n        row = loads(line)\n        require(isinstance(row, dict), \'INVALID_JSONL_ROW\')\n        rows.append(row)\n    return rows\n\n\ndef tokenize_payload(payload, tokenizer, limits):\n    prompt = tokenizer.apply_chat_template(payload[\'prompt\'], tokenize=True,\n                                            add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(payload[\'prompt\'] + payload[\'completion\'],\n                                          tokenize=True, add_generation_prompt=False)\n    require(isinstance(full, list) and isinstance(prompt, list)\n            and all(type(n) is int for n in full + prompt), \'TOKENIZER_IDS\')\n    require(full[:len(prompt)] == prompt, \'CHAT_TEMPLATE_PREFIX\')\n    completion = full[len(prompt):]\n    require(completion.count(tokenizer.eos_token_id) == 1, \'COMPLETION_EOS\')\n    eos = len(prompt) + completion.index(tokenizer.eos_token_id)\n    tail = full[eos + 1:]\n    require(eos > len(prompt), \'EMPTY_COMPLETION\')\n    require(not tail or tail == tokenizer.encode(\'\\n\', add_special_tokens=False), \'POST_EOS_TEXT\')\n    require(len(prompt) <= limits[\'max_prompt_tokens\']\n            and len(completion) <= limits[\'max_completion_tokens\']\n            and len(full) <= limits[\'max_sequence_tokens\'], \'TOKEN_BUDGET\')\n    return dict(input_ids=full, attention_mask=[1] * len(full),\n                labels=[-100] * len(prompt) + full[len(prompt):eos + 1] + [-100] * len(tail),\n                prompt_tokens=len(prompt), completion_tokens=len(completion),\n                sequence_tokens=len(full), eos_position=eos,\n                supervised_tokens=eos + 1 - len(prompt))\n\n\ndef authenticate(folder, config):\n    folder = Path(folder).resolve()\n    manifest_path = folder / \'manifest.json\'\n    require(sha_file(manifest_path) == config[\'data_manifest_sha256\'], \'MANIFEST_HASH\')\n    manifest = read_json(manifest_path)\n    for name, info in manifest[\'files\'].items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'DATA_INTEGRITY:\' + name)\n    contract = read_json(folder / \'training-contract.json\')\n    require(digest(contract) == config[\'contract_sha256\'] == manifest[\'contract_sha256\'],\n            \'CONTRACT_HASH\')\n    require(config[\'model_id\'] == contract[\'model\'][\'model_id\']\n            and config[\'revision\'] == contract[\'model\'][\'revision\']\n            == contract[\'model\'][\'tokenizer_revision\'], \'MODEL_IDENTITY\')\n    selection = read_json(folder / \'selection.json\')\n    require(set(selection[\'splits\']) == {\'train\', \'validation\'}, \'PILOT_SPLITS\')\n    ids = selection[\'splits\'][\'train\'] + selection[\'splits\'][\'validation\']\n    require(len(ids) == len(set(ids)) and not set(ids) & set(selection[\'excluded_holdout_ids\']),\n            \'DUPLICATE_OR_HOLDOUT_ID\')\n    require(manifest[\'counts\'] == {\'train\': 128, \'validation\': 32}\n            and manifest[\'holdout_payload_included\'] is False, \'PILOT_COUNTS\')\n    return contract, manifest, selection\n\n\ndef load_pilot(folder, tokenizer_folder, config):\n    from importlib.metadata import version\n    from transformers import AutoTokenizer\n    folder = Path(folder).resolve()\n    contract, manifest, selection = authenticate(folder, config)\n    reference = read_json(folder / \'tokenizer-reference.json\')\n    for name, expected in reference[\'packages\'].items():\n        require(version(name) == expected, \'TOKENIZER_LIBRARY_VERSION:\' + name)\n    for name, info in reference[\'files\'].items():\n        path = within(tokenizer_folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'TOKENIZER_ASSET:\' + name)\n    tokenizer = AutoTokenizer.from_pretrained(tokenizer_folder, local_files_only=True,\n                                               trust_remote_code=False)\n    require(tokenizer.eos_token_id == contract[\'model\'][\'eos_token_id\']\n            and tokenizer.pad_token_id == contract[\'model\'][\'pad_token_id\']\n            and hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()\n            == contract[\'model\'][\'chat_template_sha256\'], \'TOKENIZER_IDENTITY\')\n    result = {}\n    for split in (\'train\', \'validation\'):\n        tokens = read_rows(folder / (\'tokenized-\' + split + \'.jsonl\'))\n        payloads = read_rows(folder / (split + \'.jsonl\'))\n        metadata = read_rows(folder / (split + \'.metadata.jsonl\'))\n        require(len(tokens) == len(payloads) == len(metadata) == manifest[\'counts\'][split],\n                \'SPLIT_COUNT\')\n        result[split] = []\n        for index, (row, payload, meta) in enumerate(zip(tokens, payloads, metadata)):\n            rid = selection[\'splits\'][split][index]\n            require(row[\'record_id\'] == meta[\'source_record_id\'] == rid\n                    and meta[\'split\'] == split and meta[\'row_index\'] == index, \'ROW_BINDING\')\n            require(digest(payload) == meta[\'training_payload_sha256\'], \'PAYLOAD_HASH\')\n            require(row == dict(record_id=rid, **tokenize_payload(payload, tokenizer,\n                                                                contract[\'token_limits\'])),\n                    \'TOKEN_LABEL_RECOMPUTATION\')\n            # Do not allow IDs or provenance to reach model.forward().\n            result[split].append({k: row[k] for k in (\'input_ids\', \'attention_mask\', \'labels\')})\n    return result, tokenizer, contract\n', 'collator.py': '"""Strict right padding for already-audited, unshifted completion-only labels."""\nfrom dataclasses import dataclass\nfrom .data import require\n\n\n@dataclass(frozen=True)\nclass CompletionCollator:\n    pad_token_id: int = 151643\n    eos_token_id: int = 151645\n    max_sequence_tokens: int = 2048\n    pad_to_multiple_of: int = 8\n\n    def pad(self, features):\n        require(bool(features), \'EMPTY_BATCH\')\n        require(type(self.pad_to_multiple_of) is int and self.pad_to_multiple_of > 0,\n                \'PADDING_MULTIPLE\')\n        batch = {k: [] for k in (\'input_ids\', \'attention_mask\', \'labels\')}\n        for row in features:\n            require(set(row) == set(batch), \'MODEL_INPUT_FIELDS\')\n            ids, mask, labels = (row[k] for k in batch)\n            require(all(isinstance(v, list) for v in (ids, mask, labels))\n                    and len(ids) == len(mask) == len(labels) > 0, \'TOKEN_SHAPES\')\n            require(all(type(n) is int and n >= 0 for n in ids)\n                    and all(type(n) is int and n == 1 for n in mask)\n                    and all(type(n) is int for n in labels), \'TOKEN_TYPES_OR_PREPADDING\')\n            positions = [i for i, n in enumerate(labels) if n != -100]\n            require(len(positions) >= 2 and positions[0] > 0\n                    and positions == list(range(positions[0], positions[-1] + 1)),\n                    \'COMPLETION_MASK\')\n            require(all(labels[i] == ids[i] for i in positions)\n                    and labels[positions[-1]] == self.eos_token_id\n                    and sum(labels[i] == self.eos_token_id for i in positions) == 1,\n                    \'LABEL_IDS_OR_EOS\')\n        size = max(len(row[\'input_ids\']) for row in features)\n        size = ((size + self.pad_to_multiple_of - 1) // self.pad_to_multiple_of) * self.pad_to_multiple_of\n        require(size <= self.max_sequence_tokens, \'PADDED_TOKEN_LIMIT\')\n        for row in features:\n            missing = size - len(row[\'input_ids\'])\n            for key, padding in ((\'input_ids\', self.pad_token_id), (\'attention_mask\', 0), (\'labels\', -100)):\n                batch[key].append(row[key][:] + [padding] * missing)\n        return batch\n\n    def __call__(self, features):\n        import torch\n        return {key: torch.tensor(values, dtype=torch.long) for key, values in self.pad(features).items()}\n', 'runtime.py': '"""Trainer setup, update evidence and authenticated resume, independent of serving."""\nfrom datetime import datetime, timezone\nfrom importlib.metadata import distributions, version\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom .data import digest, read_json, require, sha_file, within\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,\n                                    allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef inventory(folder, excluded=()):\n    folder = Path(folder).resolve()\n    result = {}\n    for path in sorted(folder.rglob(\'*\')):\n        require(not path.is_symlink(), \'ARTIFACT_SYMLINK\')\n        if path.is_file() and path.relative_to(folder).as_posix() not in excluded:\n            result[path.relative_to(folder).as_posix()] = {\n                \'sha256\': sha_file(path), \'size_bytes\': path.stat().st_size}\n    return result\n\n\ndef verify_inventory(folder, files):\n    for name, info in files.items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'ARTIFACT_INTEGRITY:\' + name)\n\n\ndef validate_recipe(config, profile):\n    require(config[\'schema_version\'] == \'ohayo-qlora-recipe-v1\', \'RECIPE_VERSION\')\n    require(profile in (\'smoke\', \'pilot\'), \'PROFILE\')\n    require(config[\'profiles\'] == {\n        \'smoke\': {\'max_steps\': 2, \'save_steps\': 2, \'eval_steps\': 2},\n        \'pilot\': {\'max_steps\': 50, \'save_steps\': 25, \'eval_steps\': 25}}, \'STEP_PROFILES\')\n    train = config[\'training\']\n    require(train[\'per_device_train_batch_size\'] == train[\'per_device_eval_batch_size\'] == 1\n            and train[\'gradient_accumulation_steps\'] == 4, \'BATCH_RECIPE\')\n    require(all(isinstance(train[k], (float, int)) and not isinstance(train[k], bool)\n                and math.isfinite(train[k]) and train[k] > 0\n                for k in (\'learning_rate\', \'max_grad_norm\'))\n            and 0 <= train[\'warmup_ratio\'] <= 1, \'OPTIMIZER_RECIPE\')\n    require(config[\'seed\'] == 42 and config[\'lora\'] == {\n        \'r\': 8, \'lora_alpha\': 16, \'lora_dropout\': 0.05, \'bias\': \'none\',\n        \'target_modules\': \'all-linear\', \'task_type\': \'CAUSAL_LM\'}, \'LORA_RECIPE\')\n\n\ndef output_guard(output, inputs):\n    output = Path(output).resolve()\n    require(not output.exists(), \'OUTPUT_ALREADY_EXISTS\')\n    for value in inputs:\n        value = Path(value).resolve()\n        require(not output.is_relative_to(value) and not value.is_relative_to(output),\n                \'OUTPUT_OVERLAPS_INPUT\')\n    return output\n\n\ndef training_arguments(config, profile, output, *, cpu_test=False):\n    from transformers import TrainingArguments\n    values = dict(config[\'training\'])\n    args = TrainingArguments(\n        output_dir=str(output), **values, **config[\'profiles\'][profile],\n        seed=config[\'seed\'], data_seed=config[\'seed\'],\n        eval_strategy=\'steps\', save_strategy=\'steps\', logging_strategy=\'steps\', logging_steps=1,\n        save_total_limit=2, save_only_model=False, load_best_model_at_end=False,\n        fp16=not cpu_test, bf16=False, use_cpu=cpu_test,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={\'use_reentrant\': False},\n        report_to=\'none\', remove_unused_columns=False, label_names=[\'labels\'],\n        dataloader_num_workers=0, dataloader_pin_memory=not cpu_test,\n        disable_tqdm=True, save_safetensors=True, logging_nan_inf_filter=False)\n    if not cpu_test:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    return args\n\n\ndef adapter_parameters(model):\n    selected = {}\n    for name, param in model.named_parameters():\n        if param.requires_grad:\n            require(\'lora_\' in name, \'BASE_PARAMETER_TRAINABLE:\' + name)\n            selected[name] = param\n    require(bool(selected), \'NO_TRAINABLE_ADAPTER\')\n    return selected\n\n\ndef parameter_report(model):\n    selected = adapter_parameters(model)\n    return dict(trainable_parameters=sum(p.numel() for p in selected.values()),\n                total_parameter_elements=sum(p.numel() for p in model.parameters()),\n                trainable_names=sorted(selected), base_frozen=True,\n                note=\'Quantized parameter elements are storage counts, not dense base parameter counts.\')\n\n\ndef resume_checkpoint(checkpoint, expected_manifest_sha256, config, stop_after_step):\n    checkpoint = Path(checkpoint).resolve()\n    parent = checkpoint.parent\n    require(expected_manifest_sha256 is not None, \'RESUME_RECEIPT_REQUIRED\')\n    require(sha_file(parent / \'run-manifest.json\') == expected_manifest_sha256,\n            \'RESUME_MANIFEST_HASH\')\n    report = read_json(parent / \'run-manifest.json\')\n    require(report[\'status\'] == \'TRAINING_COMPLETED\' and report[\'profile\'] == \'pilot\'\n            and report[\'config_sha256\'] == digest(config), \'RESUME_RUN_IDENTITY\')\n    verify_inventory(parent, report[\'files\'])\n    require(set(inventory(parent, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n            == set(report[\'files\']), \'RESUME_EXTRA_FILES\')\n    code = {p.name: sha_file(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    require(report[\'source_files_sha256\'] == code, \'RESUME_TRAINER_CODE_CHANGED\')\n    require(checkpoint.name == \'checkpoint-25\' and checkpoint.is_dir(), \'RESUME_STEP_25_REQUIRED\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'RESUME_BINDING\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == 25,\n            \'RESUME_STATE_STEP\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\'}\n    require(required <= {p.name for p in checkpoint.iterdir()}, \'INCOMPLETE_RESUME_STATE\')\n    require(stop_after_step == 26, \'RESUME_STOP_26_REQUIRED\')\n    return checkpoint\n\n\ndef make_callbacks(output, config, *, stop_after_step=None):\n    import torch\n    from transformers import TrainerCallback\n\n    class EvidenceCallback(TrainerCallback):\n        trainer = None\n\n        def __init__(self):\n            self.before = {}\n            self.applied = 0\n            self.skipped = 0\n            self.gradient_checks = 0\n            self.started = None\n            self.start_step = None\n\n        def on_train_begin(self, args, state, control, model=None, **kwargs):\n            self.started = time.monotonic()\n            self.start_step = state.global_step\n            self.before = {n: p.detach().float().cpu().clone()\n                           for n, p in adapter_parameters(model).items()}\n\n        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):\n            gradients = [p.grad for p in adapter_parameters(model).values() if p.grad is not None]\n            require(bool(gradients), \'NO_ADAPTER_GRADIENT\')\n            require(all(torch.isfinite(g).all().item() for g in gradients), \'NONFINITE_GRADIENT\')\n            self.gradient_checks += 1\n\n        def on_step_end(self, args, state, control, **kwargs):\n            skipped = bool(self.trainer.accelerator.optimizer_step_was_skipped)\n            self.skipped += int(skipped)\n            self.applied += int(not skipped)\n            event = dict(step=state.global_step, update_applied=not skipped,\n                         elapsed_seconds=time.monotonic() - self.started)\n            with (output / \'optimizer-events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(event, allow_nan=False) + \'\\n\')\n            write_json(output / \'phase.json\', dict(phase=\'TRAINING_AND_VALIDATION\', **event))\n            if stop_after_step is not None and state.global_step >= stop_after_step:\n                control.should_training_stop = True\n                control.should_save = True\n                control.should_evaluate = True\n            return control\n\n        def on_log(self, args, state, control, logs=None, **kwargs):\n            logs = dict(logs or {})\n            for value in logs.values():\n                if isinstance(value, (int, float)):\n                    require(math.isfinite(value), \'NONFINITE_TRAINING_LOG\')\n            with (output / \'train-metrics.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(dict(step=state.global_step, **logs), allow_nan=False) + \'\\n\')\n\n        def on_save(self, args, state, control, **kwargs):\n            checkpoint = output / (\'checkpoint-\' + str(state.global_step))\n            write_json(checkpoint / \'binding.json\', {\n                \'config_sha256\': digest(config), \'data_manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'global_step\': state.global_step})\n\n        def report(self, model, global_step):\n            after = adapter_parameters(model)\n            require(set(after) == set(self.before), \'ADAPTER_PARAMETERS_CHANGED\')\n            squared = sum((p.detach().float().cpu() - self.before[n]).double().square().sum().item()\n                          for n, p in after.items())\n            return dict(start_global_step=self.start_step, global_step=global_step,\n                        attempted_updates=self.applied + self.skipped,\n                        applied_updates=self.applied, scaler_skipped_updates=self.skipped,\n                        finite_gradient_checks=self.gradient_checks,\n                        adapter_delta_l2=math.sqrt(squared), base_frozen=True)\n\n    return EvidenceCallback()\n\n\ndef fit(model, data, collator, args, output, config, *, resume=None, stop_after_step=None):\n    from transformers import Trainer\n    if not args.use_cpu:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    evidence = make_callbacks(output, config, stop_after_step=stop_after_step)\n    trainer = Trainer(model=model, args=args, train_dataset=data[\'train\'],\n                      eval_dataset=data[\'validation\'], data_collator=collator,\n                      callbacks=[evidence])\n    evidence.trainer = trainer\n    try:\n        trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    except BaseException as exc:\n        # A CUDA failure can poison the context. Do not copy tensors back to CPU\n        # while handling it, or replace the original exception with a second one.\n        failure = dict(error_type=type(exc).__name__, error=str(exc),\n                       global_step=trainer.state.global_step,\n                       observed_applied_updates=evidence.applied,\n                       observed_skipped_updates=evidence.skipped,\n                       adapter_delta_l2=None, tensor_measurement=\'NOT_ATTEMPTED_ON_FAILED_RUN\')\n        try:\n            write_json(output / \'training-failure.json\', failure)\n        except OSError:\n            pass\n        raise\n    update = evidence.report(model, trainer.state.global_step)\n    write_json(output / \'adapter-update-report.json\', update)\n    target = stop_after_step or args.max_steps\n    require(trainer.state.global_step == target and update[\'applied_updates\'] > 0\n            and update[\'scaler_skipped_updates\'] == 0\n            and update[\'attempted_updates\'] == target - update[\'start_global_step\']\n            and update[\'finite_gradient_checks\'] == update[\'attempted_updates\']\n            and math.isfinite(update[\'adapter_delta_l2\']) and update[\'adapter_delta_l2\'] > 0,\n            \'OPTIMIZER_UPDATE_PROOF_FAILED\')\n    metrics = next((entry for entry in reversed(trainer.state.log_history)\n                    if entry.get(\'step\') == trainer.state.global_step and \'eval_loss\' in entry), None)\n    if metrics is None:\n        metrics = trainer.evaluate()\n    require(math.isfinite(metrics[\'eval_loss\']), \'NONFINITE_EVAL_LOSS\')\n    write_json(output / \'eval-metrics.json\', metrics)\n    trainer.save_model(str(output / \'final-adapter\'))\n    trainer.save_state()\n    return trainer, update\n\n\ndef environment():\n    import torch\n    return dict(python=sys.version, platform=platform.platform(), torch=torch.__version__,\n                cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),\n                gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                dependencies={d.metadata[\'Name\']: d.version for d in distributions()},\n                recorded_at=datetime.now(timezone.utc).isoformat())\n\n\ndef load_training_model(config, *, allow_download=False, cache_dir=None):\n    import torch\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    require(not torch.distributed.is_initialized(), \'SINGLE_PROCESS_ONLY\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    for name, expected in config[\'dependencies\'].items():\n        require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                      bnb_4bit_use_double_quant=True,\n                                      bnb_4bit_compute_dtype=torch.float16)\n    model = AutoModelForCausalLM.from_pretrained(\n        config[\'model_id\'], revision=config[\'revision\'], trust_remote_code=False,\n        local_files_only=not allow_download, quantization_config=quantization,\n        torch_dtype=torch.float16, device_map={\'\': 0}, attn_implementation=\'sdpa\',\n        use_safetensors=True, cache_dir=str(cache_dir) if cache_dir is not None else None)\n    require(model.config._commit_hash == config[\'revision\'], \'BASE_REVISION_MISMATCH\')\n    require(getattr(model, \'is_loaded_in_4bit\', False), \'BASE_NOT_4BIT\')\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,\n                                            gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    model = get_peft_model(model, LoraConfig(**config[\'lora\'], revision=config[\'revision\']))\n    adapter_parameters(model)\n    return model\n\n\ndef finish_manifest(output, *, status, profile, config, error_type=None):\n    source = Path(__file__).parent\n    code = {p.name: sha_file(p) for p in sorted(source.glob(\'*.py\'))}\n    try:\n        git = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], capture_output=True, text=True,\n                             check=True, cwd=source).stdout.strip()\n    except (OSError, subprocess.SubprocessError):\n        git = None\n    manifest = dict(status=status, profile=profile, config_sha256=digest(config),\n                    data_manifest_sha256=config[\'data_manifest_sha256\'],\n                    source_files_sha256=code, git_head=git,\n                    note=\'Source file hashes identify uncommitted code; git_head alone does not.\',\n                    error_type=error_type,\n                    files=inventory(output, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n    write_json(output / \'run-manifest.json\', manifest)\n    files = inventory(output, (\'SHA256SUMS.txt\',))\n    (output / \'SHA256SUMS.txt\').write_text(\'\'.join(\n        info[\'sha256\'] + \'  \' + name + \'\\n\' for name, info in files.items()), encoding=\'utf-8\')\n', 'train.py': '"""CLI: CPU preflight, separate T4 smoke/pilot runs, and authenticated resume."""\nimport argparse\nfrom contextlib import redirect_stderr, redirect_stdout\nimport os\nfrom pathlib import Path\nimport sys\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import (environment, finish_manifest, fit, load_training_model, output_guard,\n                      parameter_report, resume_checkpoint, training_arguments, validate_recipe, write_json)\nfrom .storage import check_training_storage\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--bundle\', type=Path, required=True, help=\'Extracted immutable handoff root\')\n    parser.add_argument(\'--profile\', choices=(\'smoke\', \'pilot\'), required=True)\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New private run directory\')\n    parser.add_argument(\'--preflight-only\', action=\'store_true\')\n    parser.add_argument(\'--allow-model-download\', action=\'store_true\')\n    parser.add_argument(\'--cache-dir\', type=Path)\n    parser.add_argument(\'--resume-from-checkpoint\', type=Path)\n    parser.add_argument(\'--resume-manifest-sha256\')\n    parser.add_argument(\'--stop-after-step\', type=int)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\', WANDB_DISABLED=\'true\')\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    config = read_json(args.config)\n    validate_recipe(config, args.profile)\n    inputs = [args.bundle, args.config]\n    if args.resume_from_checkpoint:\n        inputs.append(args.resume_from_checkpoint.parent)\n    output = output_guard(args.output, inputs)\n    require(args.resume_from_checkpoint is not None or (args.stop_after_step is None\n            and args.resume_manifest_sha256 is None), \'RESUME_ARGUMENTS_REQUIRED\')\n    require(not args.preflight_only or args.resume_from_checkpoint is None, \'PREFLIGHT_NOT_RESUME\')\n    resume = None\n    if args.resume_from_checkpoint:\n        require(args.profile == \'pilot\', \'RESUME_PILOT_ONLY\')\n        resume = resume_checkpoint(args.resume_from_checkpoint, args.resume_manifest_sha256,\n                                    config, args.stop_after_step)\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    def phase(name):\n        write_json(output / \'phase.json\', {\'phase\': name, \'profile\': args.profile})\n    try:\n        phase(\'VALIDATING_DATA\')\n        with (output / \'stdout.txt\').open(\'w\', encoding=\'utf-8\') as stdout, \\\n             (output / \'stderr.txt\').open(\'w\', encoding=\'utf-8\') as stderr, \\\n             redirect_stdout(stdout), redirect_stderr(stderr):\n            import torch\n            from transformers import set_seed\n            set_seed(config[\'seed\'])\n            write_json(output / \'resolved-config.json\', config)\n            source = output / \'source/qlora\'\n            source.mkdir(parents=True)\n            for code in Path(__file__).parent.glob(\'*.py\'):\n                (source / code.name).write_bytes(code.read_bytes())\n            actual_environment = environment()\n            write_json(output / \'environment.json\', actual_environment)\n            (output / \'requirements-resolved.txt\').write_text(\'\'.join(\n                name + \'==\' + ver + \'\\n\' for name, ver in sorted(\n                    actual_environment[\'dependencies\'].items())), encoding=\'utf-8\')\n            write_json(output / \'data-reference.json\', {\n                \'manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'contract_sha256\': config[\'contract_sha256\'],\n                \'train\': 128, \'validation\': 32, \'holdout\': 0,\n                \'resume_checkpoint\': str(resume) if resume else None,\n                \'resume_manifest_sha256\': args.resume_manifest_sha256})\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            shapes = []\n            for split, rows in data.items():\n                for start in range(0, len(rows), 4):\n                    batch = collator(rows[start:start + 4])\n                    shapes.append(list(batch[\'input_ids\'].shape))\n            write_json(output / \'preflight.json\', {\n                \'passed\': True, \'records_checked\': sum(map(len, data.values())),\n                \'batch_shapes\': shapes, \'tensor_dtype\': \'torch.int64\',\n                \'tokenizer_recomputed\': True, \'model_weights_loaded\': False,\n                \'training\': \'NOT_RUN\'})\n            if args.preflight_only:\n                status = \'PREFLIGHT_PASSED\'\n            else:\n                from huggingface_hub.constants import HF_HUB_CACHE\n                cache = args.cache_dir or Path(HF_HUB_CACHE)\n                phase(\'CHECKING_TRAINING_STORAGE\')\n                storage = check_training_storage(config, cache, output,\n                                                   allow_download=args.allow_model_download)\n                write_json(output / \'storage-check.json\', storage)\n                require(storage[\'passed\'], \'INSUFFICIENT_DISK_FOR_PINNED_BASE_AND_RUN\')\n                phase(\'LOADING_PINNED_BASE\')\n                model = load_training_model(config, allow_download=args.allow_model_download,\n                                             cache_dir=cache)\n                base = model.get_base_model()\n                quantization = base.config.quantization_config\n                write_json(output / \'model-load-report.json\', {\n                    \'model_id\': config[\'model_id\'], \'revision\': base.config._commit_hash,\n                    \'model_class\': type(base).__name__, \'is_loaded_in_4bit\': base.is_loaded_in_4bit,\n                    \'quantization\': quantization.to_dict() if hasattr(quantization, \'to_dict\') else quantization,\n                    \'use_cache\': base.config.use_cache})\n                torch.cuda.reset_peak_memory_stats(0)\n                write_json(output / \'trainable-parameters.json\', parameter_report(model))\n                train_args = training_arguments(config, args.profile, output)\n                write_json(output / \'training-arguments.json\', train_args.to_dict())\n                write_json(output / \'training-device.json\', {\n                    \'visible_gpu_count\': train_args.n_gpu, \'world_size\': train_args.world_size,\n                    \'cuda_visible_devices\': os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                    \'mode\': \'SINGLE_GPU_NO_DATAPARALLEL\'})\n                phase(\'TRAINING_AND_VALIDATION\')\n                _, update = fit(model, data, collator, train_args, output, config,\n                                 resume=resume, stop_after_step=args.stop_after_step)\n                phase(\'SAVING_ARTIFACTS\')\n                write_json(output / \'vram.json\', {\n                    \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(0),\n                    \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(0)})\n                tokenizer.save_pretrained(output / \'tokenizer\')\n                write_json(output / \'tokenizer-reference.json\', read_json(\n                    args.bundle / \'data/pilot-v2/tokenizer-reference.json\'))\n                status = \'TRAINING_COMPLETED\'\n            phase(status)\n    except BaseException as exc:\n        error = type(exc).__name__\n        phase(\'FAILED\')\n        # Do not print payloads or secrets into public console output.\n        with (output / \'stderr.txt\').open(\'a\', encoding=\'utf-8\') as stream:\n            stream.write(error + \': \' + str(exc) + \'\\n\')\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=args.profile, config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'storage.py': '"""Budget pinned safetensors downloads and run artifacts before loading any weights."""\nfrom pathlib import Path\nimport re\nimport shutil\n\nfrom .data import read_json, require\n\nGIB = 1024 ** 3\n\n\ndef budget(missing_bytes, cache_free, output_free, same_volume):\n    require(type(missing_bytes) is int and missing_bytes >= 0, \'INVALID_WEIGHT_SIZE\')\n    cache_required = missing_bytes + (GIB if missing_bytes else 0)\n    output_required = 2 * GIB\n    if same_volume:\n        passed = min(cache_free, output_free) >= cache_required + output_required\n    else:\n        passed = cache_free >= cache_required and output_free >= output_required\n    return dict(passed=passed, missing_weight_bytes=missing_bytes,\n                cache_required_bytes=cache_required, output_required_bytes=output_required,\n                cache_free_bytes=cache_free, output_free_bytes=output_free,\n                shared_volume=same_volume,\n                note=\'Reserve 1 GiB for download overhead when needed and 2 GiB for run/checkpoint/ZIP; no base copy in the ZIP.\')\n\n\ndef check_training_storage(config, cache_dir, output, *, allow_download=False):\n    from huggingface_hub import HfApi, hf_hub_download, try_to_load_from_cache\n    cache = Path(cache_dir).expanduser().resolve()\n    output = Path(output).resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    identity = dict(repo_id=config[\'model_id\'], revision=config[\'revision\'], cache_dir=str(cache))\n    if allow_download:\n        # Metadata only: no safetensors are downloaded until the budget passes.\n        info = HfApi().model_info(config[\'model_id\'], revision=config[\'revision\'],\n                                  files_metadata=True, timeout=30)\n        require(info.sha == config[\'revision\'], \'WEIGHT_METADATA_REVISION\')\n        weights = {item.rfilename: item.size for item in info.siblings\n                   if re.fullmatch(r\'model(?:-\\d+-of-\\d+)?\\.safetensors\', item.rfilename)}\n        require(0 < len(weights) <= 16 and all(type(n) is int and n > 0 for n in weights.values()),\n                \'SAFETENSORS_SIZE_METADATA_REQUIRED\')\n    else:\n        index = hf_hub_download(filename=\'model.safetensors.index.json\',\n                                local_files_only=True, **identity)\n        names = set(read_json(index)[\'weight_map\'].values())\n        weights = {}\n        for name in names:\n            require(re.fullmatch(r\'model-\\d+-of-\\d+\\.safetensors\', name) is not None,\n                    \'WEIGHT_INDEX_FILENAME\')\n            cached = try_to_load_from_cache(filename=name, **identity)\n            require(isinstance(cached, str) and Path(cached).is_file(), \'BASE_NOT_FULLY_CACHED\')\n            weights[name] = Path(cached).stat().st_size\n    cached_bytes = 0\n    for name, size in weights.items():\n        cached = try_to_load_from_cache(filename=name, **identity)\n        if isinstance(cached, str) and Path(cached).is_file() and Path(cached).stat().st_size == size:\n            cached_bytes += size\n    total = sum(weights.values())\n    result = budget(total - cached_bytes, shutil.disk_usage(cache).free,\n                    shutil.disk_usage(output).free, cache.stat().st_dev == output.stat().st_dev)\n    return dict(result, cache_dir=str(cache), model_id=config[\'model_id\'], revision=config[\'revision\'],\n                total_weight_bytes=total, cached_weight_bytes=cached_bytes, weight_files=weights,\n                weights_downloaded_by_check=False, offline=not allow_download)\n', 'smoke.py': '"""Audit and package a completed two-step GPU smoke, without loading Qwen again."""\nimport argparse\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import digest, read_json, read_rows, require, sha_file\nfrom .runtime import inventory, verify_inventory, write_json\n\n\ndef audit(run):\n    return audit_training(run, profile=\'smoke\', steps=2)\n\n\ndef audit_training(run, *, profile, steps):\n    run = Path(run).resolve()\n    manifest = read_json(run / \'run-manifest.json\')\n    require(manifest[\'status\'] == \'TRAINING_COMPLETED\' and manifest[\'profile\'] == profile,\n            \'NOT_COMPLETED_GPU_SMOKE\')\n    verify_inventory(run, manifest[\'files\'])\n    require(set(inventory(run, (\'run-manifest.json\', \'SHA256SUMS.txt\'))) == set(manifest[\'files\']),\n            \'SMOKE_UNLISTED_FILES\')\n    sums = {}\n    for line in (run / \'SHA256SUMS.txt\').read_text(encoding=\'utf-8\').splitlines():\n        sha, name = line.split(\'  \', 1)\n        require(name not in sums, \'SMOKE_DUPLICATE_CHECKSUM\')\n        sums[name] = sha\n    require(set(sums) == set(manifest[\'files\']) | {\'run-manifest.json\'}\n            and all(sha_file(run / name) == sha for name, sha in sums.items()), \'SMOKE_CHECKSUM_LIST\')\n    config = read_json(run / \'resolved-config.json\')\n    require(digest(config) == manifest[\'config_sha256\'], \'SMOKE_CONFIG_HASH\')\n    reference = read_json(run / \'data-reference.json\')\n    require(reference[\'manifest_sha256\'] == config[\'data_manifest_sha256\']\n            == manifest[\'data_manifest_sha256\']\n            and reference[\'contract_sha256\'] == config[\'contract_sha256\']\n            and (reference[\'train\'], reference[\'validation\'], reference[\'holdout\']) == (128, 32, 0),\n            \'SMOKE_DATA_IDENTITY\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'max_steps\'] == steps and args[\'fp16\'] is True and args[\'bf16\'] is False\n            and args[\'gradient_accumulation_steps\'] == 4\n            and args[\'per_device_train_batch_size\'] == 1, \'SMOKE_TRAINING_ARGUMENTS\')\n    env = read_json(run / \'environment.json\')\n    require(env[\'cuda_available\'] is True and env[\'cuda\'] is not None\n            and len(env[\'gpus\']) == 1 and \'T4\' in env[\'gpus\'][0], \'SMOKE_GPU_EVIDENCE_REQUIRED\')\n    devices = read_json(run / \'training-device.json\')\n    require(devices[\'visible_gpu_count\'] == devices[\'world_size\'] == 1\n            and devices[\'mode\'] == \'SINGLE_GPU_NO_DATAPARALLEL\', \'SMOKE_SINGLE_GPU_REQUIRED\')\n    loaded = read_json(run / \'model-load-report.json\')\n    require(loaded[\'model_id\'] == config[\'model_id\'] and loaded[\'revision\'] == config[\'revision\']\n            and loaded[\'is_loaded_in_4bit\'] is True and loaded[\'use_cache\'] is False\n            and loaded[\'quantization\'][\'bnb_4bit_quant_type\'] == \'nf4\'\n            and loaded[\'quantization\'][\'bnb_4bit_use_double_quant\'] is True, \'SMOKE_BASE_LOAD_IDENTITY\')\n    update = read_json(run / \'adapter-update-report.json\')\n    require((update[\'start_global_step\'], update[\'global_step\'], update[\'attempted_updates\'],\n             update[\'applied_updates\'], update[\'scaler_skipped_updates\'], update[\'finite_gradient_checks\'])\n            == (0, steps, steps, steps, 0, steps) and math.isfinite(update[\'adapter_delta_l2\'])\n            and update[\'adapter_delta_l2\'] > 0 and update[\'base_frozen\'] is True,\n            \'SMOKE_OPTIMIZER_PROOF\')\n    events = read_rows(run / \'optimizer-events.jsonl\')\n    require([r[\'step\'] for r in events] == list(range(1, steps + 1))\n            and all(r[\'update_applied\'] is True for r in events), \'SMOKE_UPDATE_EVENTS\')\n    logs = read_rows(run / \'train-metrics.jsonl\')\n    loss = {r[\'step\']: r[\'loss\'] for r in logs if \'loss\' in r}\n    require(set(loss) == set(range(1, steps + 1)) and all(math.isfinite(v) for v in loss.values()), \'SMOKE_FINITE_LOSS\')\n    evaluation = read_json(run / \'eval-metrics.json\')\n    require(math.isfinite(evaluation[\'eval_loss\']), \'SMOKE_FINITE_EVAL_LOSS\')\n    require(read_json(run / \'storage-check.json\')[\'passed\'] is True, \'SMOKE_STORAGE_CHECK\')\n    parameters = read_json(run / \'trainable-parameters.json\')\n    require(parameters[\'base_frozen\'] is True and parameters[\'trainable_parameters\'] > 0,\n            \'SMOKE_BASE_FREEZE\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\', \'binding.json\'}\n    checkpoint = run / (\'checkpoint-\' + str(steps))\n    require(checkpoint.is_dir() and required <= {p.name for p in checkpoint.iterdir()},\n            \'SMOKE_CHECKPOINT_INCOMPLETE\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == steps,\n            \'SMOKE_CHECKPOINT_STEP\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'SMOKE_CHECKPOINT_BINDING\')\n    from safetensors import safe_open\n    import torch\n    with safe_open(run / \'final-adapter/adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as final, \\\n         safe_open(checkpoint / \'adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as saved:\n        require(set(final.keys()) == set(saved.keys()) and bool(final.keys()), \'SMOKE_SAVED_TENSOR_KEYS\')\n        count = 0\n        for name in final.keys():\n            require(\'lora_\' in name, \'SMOKE_UNEXPECTED_ADAPTER_TENSOR\')\n            tensor = final.get_tensor(name)\n            require(torch.isfinite(tensor).all().item() and torch.equal(tensor, saved.get_tensor(name)),\n                    \'SMOKE_SAVED_TENSORS_DIFFER\')\n            count += tensor.numel()\n    require(count == parameters[\'trainable_parameters\'], \'SMOKE_PARAMETER_COUNT\')\n    vram = read_json(run / \'vram.json\')\n    require(0 < vram[\'peak_allocated_bytes\'] <= vram[\'peak_reserved_bytes\'], \'SMOKE_VRAM_METRICS\')\n    return dict(status=\'GPU_SMOKE_PASSED\' if profile == \'smoke\' else \'GPU_PILOT_PASSED\',\n                global_step=steps, applied_updates=steps,\n                scaler_skipped_updates=0, adapter_delta_l2=update[\'adapter_delta_l2\'], base_frozen=True,\n                train_loss_by_step=loss, eval_loss=evaluation[\'eval_loss\'],\n                trainable_parameters=count, checkpoint=checkpoint.name, saved_adapter_tensors_match=True,\n                vram=vram, config_sha256=digest(config), data_manifest_sha256=config[\'data_manifest_sha256\'],\n                run_manifest_sha256=sha_file(run / \'run-manifest.json\'),\n                limitation=(\'Two-step technical smoke, not pilot 50, reload/resume verification or quality improvement.\'\n                            if profile == \'smoke\' else\n                            \'Technical pilot 50, not independent GPU reload/resume verification or quality improvement.\'))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    run = args.run.resolve()\n    report = audit(run)\n    report_path = run.with_name(run.name + \'-audit.json\')\n    if report_path.exists():\n        require(read_json(report_path) == report, \'SMOKE_AUDIT_OUTPUT_DIFFERS\')\n    else:\n        write_json(report_path, report)\n    archive = run.with_name(run.name + \'.zip\')\n    require(not archive.exists(), \'SMOKE_ZIP_EXISTS\')\n    needed = sum(p.stat().st_size for p in run.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(run.parent).free >= needed + 256 * 1024 ** 2, \'INSUFFICIENT_SPACE_FOR_SMOKE_ZIP\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(run.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(run).as_posix())\n    write_json(run.with_name(run.name + \'-receipt.json\'), dict(\n        file=archive.name, sha256=sha_file(archive), size_bytes=archive.stat().st_size,\n        audit_file=report_path.name, audit_sha256=sha_file(report_path),\n        run_manifest_sha256=report[\'run_manifest_sha256\'], model_revision=read_json(run / \'resolved-config.json\')[\'revision\'],\n        global_step=2, scope=\'PRIVATE_GPU_SMOKE_HANDOFF\', base_weights_included=False))\n    print(\'GPU_SMOKE_PASSED: \' + str(report_path))\n    print(\'Private artifact: \' + str(archive))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'}
SMOKE_SOURCE_ID = 'a31be6f399b3f3af864b2fe2348ac8f229572d64454e4119bfc8843b48e376ac'
assert hashlib.sha256(json.dumps(SMOKE_SOURCES, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest() == SMOKE_SOURCE_ID
BASE_TOOLS = RUN_TOOLS
SMOKE_TOOLS = WORK / ('tools-smoke-' + SMOKE_SOURCE_ID[:12])
if not SMOKE_TOOLS.exists():
    shutil.copytree(BASE_TOOLS, SMOKE_TOOLS)
    for name, content in SMOKE_SOURCES.items():
        (SMOKE_TOOLS / 'scripts/qlora' / name).write_bytes(content.encode('utf-8'))
expected_files = {p.relative_to(BASE_TOOLS).as_posix() for p in BASE_TOOLS.rglob('*') if p.is_file()} | {'scripts/qlora/' + name for name in SMOKE_SOURCES}
assert {p.relative_to(SMOKE_TOOLS).as_posix() for p in SMOKE_TOOLS.rglob('*') if p.is_file()} == expected_files
for relative in expected_files:
    path = SMOKE_TOOLS / relative
    if relative.startswith('scripts/qlora/') and Path(relative).name in SMOKE_SOURCES:
        assert path.read_bytes() == SMOKE_SOURCES[Path(relative).name].encode('utf-8')
    else:
        assert path.read_bytes() == (BASE_TOOLS / relative).read_bytes()
SMOKE_DIR = WORK / 'runs' / ('smoke-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
BASE_CACHE = WORK / 'base-cache'
TRAIN_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0', PYTHONDONTWRITEBYTECODE='1', PYTHONUTF8='1', TOKENIZERS_PARALLELISM='false', HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0', HF_XET_CHUNK_CACHE_SIZE_BYTES='0', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
command = [str(VENV / 'bin/python'), '-B', '-m', 'scripts.qlora.train', '--bundle', str(HANDOFF), '--config', str(SMOKE_TOOLS / 'configs/qlora-pilot.json'), '--profile', 'smoke', '--output', str(SMOKE_DIR), '--cache-dir', str(BASE_CACHE), '--allow-model-download']
print('Smoke: 2 optimizer steps. First run may download base weights; pilot 50 is NOT started.', flush=True)
print('Run:', SMOKE_DIR, flush=True)
SMOKE_PROCESS = subprocess.Popen(command, cwd=SMOKE_TOOLS, env=TRAIN_ENV)
started = time.monotonic()
try:
    while SMOKE_PROCESS.poll() is None:
        phase = {}
        try:
            phase = json.loads((SMOKE_DIR / 'phase.json').read_text())
        except (FileNotFoundError, ValueError):
            pass
        cached = sum(p.stat().st_size for p in BASE_CACHE.rglob('*') if p.is_file() and not p.is_symlink()) if BASE_CACHE.exists() else 0
        step = phase.get('step', '-')
        print(f"{int(time.monotonic() - started)}s | {phase.get('phase', 'STARTING')} | step={step} | cache={cached / 1024**3:.2f} GiB", flush=True)
        time.sleep(15)
except BaseException:
    SMOKE_PROCESS.terminate()
    try:
        SMOKE_PROCESS.wait(timeout=30)
    except subprocess.TimeoutExpired:
        SMOKE_PROCESS.kill()
        SMOKE_PROCESS.wait()
    raise
if SMOKE_PROCESS.returncode != 0:
    for name in ('storage-check.json', 'stderr.txt'):
        path = SMOKE_DIR / name
        if path.exists():
            print(name + ':\n' + path.read_text(encoding='utf-8')[-6000:])
    raise RuntimeError('Smoke failed; keep this run and send the error output. Do not run pilot.')
print('Training process finished. Run the next cell to audit and export evidence.')


In [ ]:
assert SMOKE_PROCESS.returncode == 0, 'Smoke has not completed successfully.'
subprocess.run([str(VENV / 'bin/python'), '-B', '-m', 'scripts.qlora.smoke', '--run', str(SMOKE_DIR)], cwd=SMOKE_TOOLS, env=TRAIN_ENV, check=True)
SMOKE_AUDIT = SMOKE_DIR.with_name(SMOKE_DIR.name + '-audit.json')
print(SMOKE_AUDIT.read_text(encoding='utf-8'))
print('ZIP:', SMOKE_DIR.with_suffix('.zip'))
print('Receipt:', SMOKE_DIR.with_name(SMOKE_DIR.name + '-receipt.json'))
print('Send the audit JSON, ZIP and receipt. Do not run pilot 50 yet.')
